# Day 10 — Review + project: a GenAI content assistant

**Phase 1 · Generative AI Foundations · ~45 min · Needs a provider**

## 🎯 Objective
Combine everything from Phase 1 — prompting, few-shot, structured output — into one
useful tool, and set up the leap into **agents**.

## 🧠 The bridge to Phase 2
You can now make a model *produce* reliable, structured results. An **agent** is the
next step: a model that doesn't just produce text, but **decides to take actions**
(call tools) in a loop. Today's assistant is the perfect "last GenAI, first agent"
artifact — it returns a structured plan you could hand to tools tomorrow.

```mermaid
flowchart LR
    T["Any text"] --> A["🧠 analyze()"] --> J["{summary, sentiment, keywords}"]
```

## 🔬 Exercise
Implement `analyze(text)` → a validated object with `summary` (str),
`sentiment` (POSITIVE/NEGATIVE/NEUTRAL), and `keywords` (list of str). Use a clear
system prompt + the JSON validate/repair pattern from Day 7.

## ✅ Done when
- `analyze()` returns a typed result for any input text.

## 🎉 Phase 1 complete!
You understand generative AI: how it works, how to steer it, and how to get
trustworthy structured output. Tick Days 1–10 in the [handbook](../../README.md).

## 🔭 Next: Phase 2 — From Generative to Agentic
Now we give the model a **loop**, **tools**, and **autonomy**. Same model — but it
starts to *act*.


### ▶ Run this first

In [ ]:
# Run me first: make the course's shared/ package importable from this notebook.
import sys, pathlib
for _cand in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_cand / 'shared' / 'llm.py').exists():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        break


## 🔬 Your turn
Complete the `TODO`s below, then run the cell (**Shift+Enter**).

In [ ]:
import json

from pydantic import BaseModel, ValidationError

from shared.llm import chat


class Analysis(BaseModel):
    summary: str
    sentiment: str        # POSITIVE | NEGATIVE | NEUTRAL
    keywords: list[str]


def _strip_fences(s):
    s = s.strip()
    if s.startswith("```"):
        s = s.strip("`")
        if "\n" in s:
            s = s.split("\n", 1)[1]
    return s.strip()


def analyze(text, attempts=2):
    """Return a validated Analysis for `text`.

    TODO:
      - system prompt: return ONLY JSON {summary, sentiment (POSITIVE/NEGATIVE/NEUTRAL),
        keywords (array of strings)}
      - call chat(..., temperature=0), then parse+validate with Analysis,
        repairing on failure (like Day 7).
    """
    raise NotImplementedError


SAMPLE = ("The new update is fast and the UI is gorgeous, but it crashed twice "
          "and support never replied.")

if __name__ == "__main__":
    print(analyze(SAMPLE))


---
---
## 🔒 Solution
*Try it yourself first!* Run the cell below only when you're ready to compare.

In [ ]:
import json

from pydantic import BaseModel, ValidationError

from shared.llm import chat


class Analysis(BaseModel):
    summary: str
    sentiment: str
    keywords: list[str]


def _strip_fences(s):
    s = s.strip()
    if s.startswith("```"):
        s = s.strip("`")
        if "\n" in s:
            s = s.split("\n", 1)[1]
    return s.strip()


SYSTEM = (
    "Analyze the user's text. Reply with ONLY JSON with keys: "
    "summary (one sentence), sentiment (one of POSITIVE, NEGATIVE, NEUTRAL), "
    "keywords (array of 3-5 short strings)."
)


def analyze(text, attempts=2):
    raw = chat([{"role": "system", "content": SYSTEM},
                {"role": "user", "content": text}], temperature=0)
    for _ in range(attempts + 1):
        try:
            return Analysis(**json.loads(_strip_fences(raw)))
        except (json.JSONDecodeError, ValidationError) as exc:
            raw = chat([{"role": "system", "content": "Fix into valid JSON for keys "
                         "summary, sentiment, keywords. ONLY JSON."},
                        {"role": "user", "content": f"Error: {exc}\nBroken: {raw}"}],
                       temperature=0)
    raise ValueError("Could not produce valid JSON.")


SAMPLE = ("The new update is fast and the UI is gorgeous, but it crashed twice "
          "and support never replied.")

if __name__ == "__main__":
    print(analyze(SAMPLE))
